# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR² dataset with [mlcroissant](https://github.com/mlcommons/croissant). You will load the dataset via its Croissant schema, review its structure, extract and manipulate data, and visualize it using pandas and matplotlib.

### Dataset Source
<br>
**Croissant schema URL:**
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata and records from the Croissant dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published on: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}\n")

## 2. Data Overview

Review available record sets (tables), fields (columns), and their `@id`s. All references use their Croissant `@id` for clarity and reproducibility.

In [ ]:
# List record sets and their @id, fields, and field @id in the dataset
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("------------------------")

## 3. Data Extraction

Load data from each record set using their `@id`. You can use the fields listed above for further selection or processing. The result for each set is a pandas DataFrame.

In [ ]:
# Extract data from each record set into a DataFrame
dataframes = {}

# Gather list of all record_set @id
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Record set @ids: {record_set_ids}\n")

# You can set this to a specific record set @id to focus your analysis
selected_record_set_id = record_set_ids[0] if record_set_ids else None

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

if selected_record_set_id:
    print(f"\nFields (columns) in record set '{selected_record_set_id}':")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No record sets available in this dataset.")

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing: filtering, normalization, and grouping by provided fields using their `@id` as the column name (as per Croissant specification).

In [ ]:
# EDA: Filter, normalize and group using field @id
import numpy as np

if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"Column names in selected record set: {df.columns.tolist()}\n")

    # Attempt auto-detection: find first numeric field
    numeric_field_id = None
    group_field_id = None
    for c in df.columns:
        if np.issubdtype(df[c].dtype, np.number):
            numeric_field_id = c
            break

    # Attempt auto-detection: first non-numeric field for grouping
    for c in df.columns:
        if not np.issubdtype(df[c].dtype, np.number):
            group_field_id = c
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtering rows where {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records left\n")

        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"First 5 entries with normalized '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Grouping (if suitable field found)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped by '{group_field_id}' (mean {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric fields available for EDA in this record set.")
else:
    print("No record set selected for EDA.")

## 5. Visualization

Visualize data distributions and relationships between fields. This example plots the distribution of the selected numeric field and, if grouped, shows group means.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if selected_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped, plot group means
    if group_field_id and group_field_id in df.columns:
        means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)[:10]
        plt.figure(figsize=(10, 5))
        means.plot(kind="bar")
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 10 groups)")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("Unable to plot distributions: missing numeric field or data.")

## 6. Conclusion

- This notebook showed how to access and analyze the FAIR² mass spectrometry dataset using references by Croissant `@id` throughout.
- We used `mlcroissant` to inspect metadata, extract tables, and process fields based on their semantic identifiers.
- Results can be further extended by focusing on additional record sets or custom field operations. 

For detailed information or support on the Croissant format, see: https://mlcommons.org/croissant/.
